# 09 — Sensitivity and robustness summary

Compares segmentation profiles, encodings, one-recording-per-participant estimates, and exact-session Rest context.

This notebook saves its visual summary and audit tables into separate `figures/` and `tables/` directories. Its final cell states the main output, any decision required, and whether the next stage is allowed.

In [ ]:
from pathlib import Path
import json
import shutil
import subprocess
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml
from IPython.display import Image, Markdown, display

def find_project_root():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "paper1_qc").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the paper_1 project.")

ROOT = find_project_root()
CONFIG = ROOT / "config" / "project.yaml"
OUTPUT = ROOT / "outputs"
MAIN_OUTPUTS = ROOT / "MAIN outputs"
MAIN_OUTPUTS.mkdir(exist_ok=True)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

def run_cli(*arguments):
    command = [sys.executable, "-m", "paper1_qc.cli", "--config", str(CONFIG), *arguments]
    print("RUN:", " ".join(map(str, command)))
    subprocess.run(command, cwd=ROOT, check=True)

def read_table(path_without_suffix):
    stem = Path(path_without_suffix)
    parquet = stem.with_suffix(".parquet")
    csv = stem.with_suffix(".csv")
    if parquet.exists():
        return pd.read_parquet(parquet)
    if csv.exists():
        try:
            return pd.read_csv(csv)
        except pd.errors.EmptyDataError:
            return pd.DataFrame()
    raise FileNotFoundError(f"Missing table: {parquet} or {csv}")

def stage_directories(relative_stage):
    stage = OUTPUT / relative_stage
    figures = stage / "figures"
    tables = stage / "tables"
    figures.mkdir(parents=True, exist_ok=True)
    tables.mkdir(parents=True, exist_ok=True)
    return stage, figures, tables

def save_table(frame, directory, name):
    path = Path(directory) / f"{name}.csv"
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print("TABLE:", path.relative_to(ROOT), f"({len(frame):,} rows)")
    return path

def save_figure(fig, directory, name):
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    png = directory / f"{name}.png"
    svg = directory / f"{name}.svg"
    fig.savefig(png, bbox_inches="tight")
    fig.savefig(svg, bbox_inches="tight")
    print("FIGURE:", png.relative_to(ROOT))
    return png, svg

def stage_gate(stage_name, can_continue, reasons, next_step):
    status = "PASS — safe to continue" if can_continue else "BLOCKED — decision/action required"
    color = "#1B7F3A" if can_continue else "#B22222"
    details = "\n".join(f"- {reason}" for reason in reasons) if reasons else "- No blocking findings."
    display(Markdown(
        f"### {stage_name}: <span style='color:{color}'>{status}</span>\n\n"
        f"{details}\n\n**Next step:** {next_step}"
    ))
    return can_continue

assert CONFIG.exists(), "Copy config/project.example.yaml to config/project.yaml and review it."
print("Project:", ROOT)
print("Config:", CONFIG)
print("MAIN outputs:", MAIN_OUTPUTS)


## Main output and decisions

Main output: a visual robustness dashboard and the underlying sensitivity tables. Any conclusion that changes sign or loses adequate support must be reported as sensitive rather than hidden.

In [ ]:
RUN_CONSERVATIVE = False  # Change to True only when you intend to run this stage.

if RUN_CONSERVATIVE:
    run_cli('extract', '--profile', 'conservative')
else:
    print('Conservative profile not run.')

In [ ]:
RUN_PERMISSIVE = False  # Change to True only when you intend to run this stage.

if RUN_PERMISSIVE:
    run_cli('extract', '--profile', 'permissive')
else:
    print('Permissive profile not run.')

In [ ]:
RUN_SENSITIVITY = False  # Change to True only when you intend to run this stage.

if RUN_SENSITIVITY:
    run_cli('sensitivity')
else:
    print('Sensitivity aggregation not run.')

In [ ]:
RUN_ENCODING = False  # Change to True only when you intend to run this stage.

if RUN_ENCODING:
    run_cli('encoding-sensitivity')
else:
    print('Encoding sensitivity not run.')

In [ ]:
RUN_REST = False  # Change to True only when you intend to run this stage.

if RUN_REST:
    run_cli('rest-reference')
else:
    print('Rest-reference sensitivity not run.')

In [ ]:
STAGE, FIGURES, TABLES = stage_directories(Path("04_analysis") / "sensitivity_summary")
profile = read_table(OUTPUT / "04_analysis" / "sensitivity" / "segmentation_profile_robustness")
encoding = read_table(OUTPUT / "04_analysis" / "encoding_sensitivity" / "paired_encoding_robustness")
rest = read_table(OUTPUT / "04_analysis" / "rest_reference" / "rest_reference_summary")
save_table(profile, TABLES, "segmentation_profile_robustness")
save_table(encoding, TABLES, "encoding_robustness")
save_table(rest, TABLES, "rest_reference_summary")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
if not profile.empty:
    sns.barplot(data=profile, x="profile", y="spearman_rho", hue="feature", ax=axes[0])
    axes[0].legend([], [], frameon=False)
    axes[0].set(title="Segmentation-profile robustness", ylabel="Spearman rho vs primary")
if not encoding.empty:
    plot = encoding.sort_values("spearman_rho").tail(20)
    sns.barplot(data=plot, x="spearman_rho", y="feature", color="#4C78A8", ax=axes[1])
    axes[1].set(title="WAV/WEBM metric agreement", xlabel="Spearman rho", ylabel="")
fig.tight_layout()
save_figure(fig, FIGURES, "sensitivity_dashboard")
plt.show()

missing_outputs = [
    name for name, frame in {
        "segmentation profiles": profile,
        "encoding comparison": encoding,
        "Rest reference": rest,
    }.items() if frame.empty
]
sensitivity_ready = stage_gate(
    "Sensitivity summary",
    not missing_outputs,
    ["Missing output: " + name for name in missing_outputs],
    "Proceed to manuscript tables/figures only after all prespecified sensitivities exist.",
)